In [10]:
# Golden Gate Bridge steering with pretrained SAE

import torch
from transformer_lens import HookedTransformer
from sae_lens import SAE

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)

sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.8.hook_resid_pre",
    device=device,
)

# Find feature specific to Golden Gate (high on concept, low on baseline)
concept_prompts = [
    "The Golden Gate Bridge spans the San Francisco Bay",
    "Tourists photograph the Golden Gate Bridge in the fog",
    "The Golden Gate Bridge was painted international orange",
    "Walking across the Golden Gate Bridge takes thirty minutes",
    "The Golden Gate Bridge connects San Francisco to Marin County",
]
baseline_prompts = [
    "The best recipe for chocolate cake requires butter",
    "Dogs are popular pets around the world today",
    "The stock market opened higher this morning",
    "Basketball players train every day for hours",
    "Students study mathematics and science in school",
]

def get_mean_features(prompts):
    all_feats = []
    with torch.no_grad():
        for p in prompts:
            tokens = model.to_tokens(p)
            _, cache = model.run_with_cache(tokens)
            resid = cache["blocks.8.hook_resid_pre"][0, -1]
            all_feats.append(sae.encode(resid))
    return torch.stack(all_feats).mean(dim=0)

concept_feats = get_mean_features(concept_prompts)
baseline_feats = get_mean_features(baseline_prompts)
diff = concept_feats - baseline_feats
best_feature = diff.argmax().item()

lines = [f"Golden Gate feature: {best_feature} (diff: {diff[best_feature]:.2f}, concept: {concept_feats[best_feature]:.2f}, baseline: {baseline_feats[best_feature]:.2f})", ""]

# Steering hook
def steer_hook(resid, hook, feature_id=best_feature, strength=5.0):
    resid[:, -1] += sae.W_dec[feature_id] * strength
    return resid

prompt = "Here is the title of a love story: "

# No steering
out = model.generate(prompt, max_new_tokens=30, temperature=0)
lines.append(f"No steering:\n{out}\n")

# With steering
for s in [30, 35, 40, 45, 50]:
    hook_fn = lambda resid, hook, s=s: steer_hook(resid, hook, strength=s)
    with model.hooks(fwd_hooks=[("blocks.8.hook_resid_pre", hook_fn)]):
        out = model.generate(prompt, max_new_tokens=30, temperature=0)
    lines.append(f"Strength={s}:\n{out}\n")

print("\n".join(lines))

Loaded pretrained model gpt2 into HookedTransformer


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

Golden Gate feature: 23937 (diff: 11.40, concept: 11.40, baseline: 0.00)

No steering:
Here is the title of a love story:  I'm a little bit of a fan of the original series, but I'm not sure if I'm going to be able to enjoy it.

Strength=30:
Here is the title of a love story:  "The Story of the Little Mermaid" by Anne Frank.
The story of the Little Mermaid is a story of love, romance, and the

Strength=35:
Here is the title of a love story:  "The Story of the Little Mermaid" by Anne Frank.
The story of the Little Mermaid is a story of love, romance, and the

Strength=40:
Here is the title of a love story:  "The Story of the New York Times."
The story of the New York Times is a story of the New York Times. It is a

Strength=45:
Here is the title of a love story:  "The Story of the City of San Francisco" by David A. Levine.
Bay Area News Area
Bay Area News Area
Bay Area

Strength=50:
Here is the title of a love story:  "The Story of the City of San Francisco" by David A. Levine.
Bay Area 